# FreshGuard — Notebook 3 / 5: Train YOLO26s Detector (24 classes)

**Purpose.** Fine‑tune Ultralytics YOLO26**s** on the 24‑class detection dataset produced by notebook 2. One forward pass at inference returns boxes labelled with both produce type *and* freshness — no two‑stage cascade.

**Why YOLO26s.** From the [Ultralytics YOLO26 docs](https://docs.ultralytics.com/models/yolo26/) (Jan 2026 release): NMS‑free end‑to‑end head, ProgLoss + STAL improving small‑object recall, MuSGD optimizer recommended for longer runs. The `s` variant scores 48.6 mAP50‑95 on COCO with only 9.5 M params — best accuracy/compute tradeoff for this 24‑class task.

**Imbalance handling.** The dataset has a 41:1 class imbalance. We set `cls_pw=1.0` per the docs ("increase to 1.0 for severe cases") and rely on the per‑class report at the end to flag minority classes that need data work.

**Inputs:**
- `<your-handle>/freshguard-detector-data` — from notebook 2

**Outputs (publish as `<your-handle>/freshguard-yolo26s`):**
- `yolo26s_food_freshness.pt` — best checkpoint by val mAP50‑95
- `detector_metrics.json` — per‑class P/R/mAP
- training plots (results.png, confusion matrix, label distribution)

**Expected runtime.** ≈3–6 hours on a Kaggle T4 ×2 for the full 200 epochs. Always run the smoke‑test cell first (`epochs=2 fraction=0.05` ≈ 5 min) before committing the full run.

In [ ]:
!pip install --quiet "ultralytics>=8.3.0,<9" pyarrow

In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO

# ---------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------
DATASET_YAML = Path("/kaggle/input/datasets/elsmmany/freshguard-detector-data/detector/detector_dataset.yaml")
OUTPUT_DIR = Path("/kaggle/working/yolo26s_run")
BASE_MODEL = "yolo26s.pt"  # Ultralytics auto-downloads pretrained COCO weights
RUN_NAME = "freshguard"

# Set to True for a 5-minute smoke run before the full multi-hour commit.
SMOKE_TEST = True

# Full-run hyperparameters (used when SMOKE_TEST is False).
FULL_EPOCHS = 150  # ~10 hrs at 4 min/epoch on 2x T4; fits Kaggle's 12-hour cap with buffer
FULL_FRACTION = 1.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
n_gpus = torch.cuda.device_count()
if n_gpus >= 2:
    device = list(range(n_gpus))
elif n_gpus == 1:
    device = 0
else:
    device = "cpu"
print(f"Detected {n_gpus} GPU(s). Training device(s): {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Training will be very slow.")

if not DATASET_YAML.exists():
    raise FileNotFoundError(
        f"Detector dataset not found at {DATASET_YAML}. "
        "Attach `<your-handle>/freshguard-detector-data` as input."
    )
# detector_dataset.yaml was written by notebook 2 with `path: /kaggle/working/detector`,
# which only existed in that notebook's working directory. Rewrite it on the fly so
# YOLO can find the actual images at the freshguard-detector-data mount.
import yaml as _yaml
DATASET_DIR = DATASET_YAML.parent
spec = _yaml.safe_load(DATASET_YAML.read_text())
spec["path"] = str(DATASET_DIR)
PATCHED_YAML = OUTPUT_DIR / "detector_dataset.yaml"
PATCHED_YAML.write_text(_yaml.safe_dump(spec, sort_keys=False))
print(f"Rewrote dataset YAML path -> {DATASET_DIR}")
print(f"Using patched YAML: {PATCHED_YAML}")
DATASET_YAML = PATCHED_YAML
print(f"Dataset: {DATASET_YAML}")
print(f"Output:  {OUTPUT_DIR}")

## Training hyperparameters

Sourced from the Ultralytics YOLO26 docs and training guide (scraped 2026‑05‑02). Augmentation is tuned for produce: no vertical flip (orientation matters), modest mixup to combat overfit on minority classes, mosaic on for 190 epochs then off for the final 10 (Ultralytics standard).

Don't change these without good reason — the defaults are the result of explicit Ultralytics recommendations for this scale of run.

In [ ]:
# Ultralytics rejects batch=-1 (AutoBatch) under multi-GPU. Use 8 per GPU as
# a safe default; bump to 12 or 16 per GPU if you have memory headroom.
PER_GPU_BATCH = 8
if isinstance(device, list):
    BATCH_SIZE = PER_GPU_BATCH * len(device)
elif device == "cpu":
    BATCH_SIZE = 4
else:
    BATCH_SIZE = -1  # autobatch on single GPU
print(f"Using BATCH_SIZE={BATCH_SIZE}")

TRAIN_KWARGS = dict(
    data=str(DATASET_YAML),
    epochs=2 if SMOKE_TEST else FULL_EPOCHS,
    time=None if SMOKE_TEST else 10.5,  # hard cap (hrs) so wall-time stays inside Kaggle's 12 hr limit
    fraction=0.05 if SMOKE_TEST else FULL_FRACTION,
    patience=30,
    batch=BATCH_SIZE,         # multi-GPU requires explicit batch (multiple of n_gpus)
    imgsz=640,
    optimizer="MuSGD",        # Ultralytics: "Recommended for longer YOLO26 runs"
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=True,
    amp=True,                 # mixed precision
    close_mosaic=10,          # disable mosaic for the last 10 epochs
    cls_pw=1.0,               # SEVERE imbalance (41:1) — docs recommend 1.0 for severe cases
    # Augmentation
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5,
    shear=0.0, perspective=0.0,
    fliplr=0.5,
    flipud=0.0,               # do NOT vertical-flip — produce orientation is informative
    mosaic=1.0, mixup=0.15, cutmix=0.0, copy_paste=0.0,
    # Reproducibility
    seed=0,
    deterministic=True,
    plots=True,
    save=True,
    project=str(OUTPUT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    device=device,
)
for key in ("epochs", "fraction", "optimizer", "cls_pw", "close_mosaic", "mosaic", "mixup", "flipud"):
    print(f"  {key:>14}: {TRAIN_KWARGS[key]}")
if SMOKE_TEST:
    print("\n>>> SMOKE TEST mode — set SMOKE_TEST = False above for the full run.")

## Train

In [ ]:
model = YOLO(BASE_MODEL)
results = model.train(**TRAIN_KWARGS)
print("Training complete.")
# Ultralytics returns None from .train() under some DDP configs;
# fall back to the deterministic project/name path.
RUN_DIR = Path(results.save_dir) if (results is not None and hasattr(results, "save_dir")) else (OUTPUT_DIR / RUN_NAME)
print(f"Best weights: {RUN_DIR}/weights/best.pt")

## Validate on the test split + emit metrics JSON

We re‑run validation on the test split explicitly so the reported numbers come from data the model never saw during training or model selection (which uses the val split).

In [ ]:
best_path = RUN_DIR / "weights" / "best.pt"
best_model = YOLO(str(best_path))

metrics = best_model.val(data=str(DATASET_YAML), split="test", verbose=False)

names = list(best_model.names.values()) if isinstance(best_model.names, dict) else list(best_model.names)
per_class = {}
if hasattr(metrics.box, "maps") and metrics.box.maps is not None:
    for i, ap in enumerate(metrics.box.maps):
        per_class[names[i]] = {"mAP50_95": float(ap)}
if hasattr(metrics.box, "p") and metrics.box.p is not None:
    for i, p in enumerate(metrics.box.p):
        per_class.setdefault(names[i], {})["precision"] = float(p)
if hasattr(metrics.box, "r") and metrics.box.r is not None:
    for i, r in enumerate(metrics.box.r):
        per_class.setdefault(names[i], {})["recall"] = float(r)
if hasattr(metrics.box, "ap50") and metrics.box.ap50 is not None:
    for i, ap50 in enumerate(metrics.box.ap50):
        per_class.setdefault(names[i], {})["mAP50"] = float(ap50)

metrics_payload = {
    "model": BASE_MODEL,
    "checkpoint": str(best_path),
    "split": "test",
    "overall": {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
    },
    "macro": {
        "mAP50": (sum(c.get("mAP50", 0.0) for c in per_class.values()) / max(1, len(per_class))),
        "mAP50_95": (sum(c["mAP50_95"] for c in per_class.values()) / max(1, len(per_class))),
    },
    "per_class": per_class,
    "hyperparameters": {k: v for k, v in TRAIN_KWARGS.items() if k != "data"},
}

metrics_path = OUTPUT_DIR / "detector_metrics.json"
metrics_path.write_text(json.dumps(metrics_payload, indent=2))
print(f"Wrote {metrics_path}")

print(f"\nOverall (test): P={metrics_payload['overall']['precision']:.3f} "
      f"R={metrics_payload['overall']['recall']:.3f} "
      f"mAP50={metrics_payload['overall']['mAP50']:.3f} "
      f"mAP50-95={metrics_payload['overall']['mAP50_95']:.3f}")
print(f"Macro mAP50: {metrics_payload['macro']['mAP50']:.3f}")

minority = [c for c in per_class if c.startswith(("bitter_gourd", "okra", "mango", "strawberry"))]
if minority:
    print("\nMinority class metrics:")
    for c in minority:
        m = per_class[c]
        print(f"  {c:25s} P={m.get('precision', 0):.3f} R={m.get('recall', 0):.3f} mAP50={m.get('mAP50', 0):.3f}")

## Stage the artifacts for publishing

Copy the final checkpoint and the metrics JSON to a clean `artifacts/` directory so the published Kaggle Dataset only contains what the next notebooks need.

In [ ]:
publish_dir = Path("/kaggle/working/freshguard-yolo26s")
publish_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_path, publish_dir / "yolo26s_food_freshness.pt")
shutil.copy2(metrics_path, publish_dir / "detector_metrics.json")
for plot in ("results.png", "confusion_matrix.png", "labels.jpg", "PR_curve.png"):
    src = RUN_DIR / plot
    if src.exists():
        shutil.copy2(src, publish_dir / plot)
print("Files staged for the Kaggle Dataset:")
for p in sorted(publish_dir.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024 / 1024:.2f} MiB)")

## Publish as a Kaggle Dataset

1. Set `SMOKE_TEST = False` and **Save Version → Save & Run All (Commit)**. The full run takes 3–6 hours.
2. From the notebook **Output** tab, **New Dataset** → name it `freshguard-yolo26s`.
3. Notebook 5 (eval) attaches this dataset. The local Streamlit app gets the same `.pt` file via `scripts/download_artifacts.py` once it's mirrored to a GitHub Release.

**Verification before publishing:**
- `metrics.overall.recall ≥ 0.55` (vs. the previous 0.438) and `mAP50 ≥ 0.50`.
- Per‑class minority recall (Bittergroud, Okra, Mango, Strawberry) is reported and is non‑zero.
- `results.png` shows train and val curves converging without obvious overfit (val loss not climbing in the last 30 epochs).